###Phase 1 – Data Understanding

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


### 1 Load data


In [0]:
spark = SparkSession.builder.getOrCreate()


In [0]:
customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

In [0]:
cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

In [0]:
sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

In [0]:
sales = sales.withColumn("sale_date", to_date(col("sale_date")))


### cleaning

In [0]:
customers.fillna({'name':'unknown'}).display()

customer_id,name,city
1,John,Hyderabad
2,Alice,Chennai
3,unknown,Bangalore


In [0]:
cars = cars.withColumn(
    "price",
    when(col("price") < 0, abs(col("price"))).otherwise(col("price"))
)

In [0]:
df = customers.join(sales, "customer_id", "left") \
              .join(cars, "car_id", "left")

In [0]:
df.display()

car_id,customer_id,name,city,sale_id,sale_date,quantity,brand,model,price
101,1,John,Hyderabad,1,2024-01-01,1,Toyota,Camry,30000
102,2,Alice,Chennai,2,2024-01-02,2,Honda,Civic,20000
null,3,null,Bangalore,null,null,null,null,null,null


In [0]:
df = df.fillna({
    "name": "Unknown",
    "car_id": 0,
    "sale_id": 0,
    "quantity": 0,
    "price": 0,
    "brand": "Unknown",
    "model": "Unknown"
}).withColumn("sale_date", coalesce(col("sale_date"), lit("1900-01-01").cast("date")))

In [0]:
df.display()

car_id,customer_id,name,city,sale_id,sale_date,quantity,brand,model,price
101,1,John,Hyderabad,1,2024-01-01,1,Toyota,Camry,30000
102,2,Alice,Chennai,2,2024-01-02,2,Honda,Civic,20000
0,3,Unknown,Bangalore,0,1900-01-01,0,Unknown,Unknown,0


In [0]:
df.groupBy("customer_id").sum("price").show()

+-----------+----------+
|customer_id|sum(price)|
+-----------+----------+
|          1|     30000|
|          2|     20000|
|          3|         0|
+-----------+----------+



In [0]:
df.display()

car_id,customer_id,name,city,sale_id,sale_date,quantity,brand,model,price
101,1,John,Hyderabad,1,2024-01-01,1,Toyota,Camry,30000
102,2,Alice,Chennai,2,2024-01-02,2,Honda,Civic,20000
0,3,Unknown,Bangalore,0,1900-01-01,0,Unknown,Unknown,0


### Phase 3 – Validation



In [0]:
#anti join
invalid_customers = sales.join(
    customers,"customer_id","left_anti"
)

invalid_customers.show()

+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



In [0]:
invalid_cars = sales.join(cars, "car_id", "left_anti")

In [0]:
from pyspark.sql.functions import col, count, when

null_report = customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in customers.columns
])

In [0]:
negative_price = cars.filter(col("price") < 0)

In [0]:
print(" VALIDATION REPORT")
print("Invalid Customers:", invalid_customers.count())
print("Invalid Cars:", invalid_cars.count())
print("Customers with Null Names:", customers.filter(col("name").isNull()).count())
print("Negative Price Records:", negative_price.count())

 VALIDATION REPORT
Invalid Customers: 1
Invalid Cars: 0
Customers with Null Names: 1
Negative Price Records: 0


### Phase 4 – Transformation


1 Revenue per customer


In [0]:
df = df.withColumn(
    "revenue",
    col("price") * col("quantity")
)
df.show()

+------+-----------+-------+---------+-------+----------+--------+-------+-------+-----+-------+
|car_id|customer_id|   name|     city|sale_id| sale_date|quantity|  brand|  model|price|revenue|
+------+-----------+-------+---------+-------+----------+--------+-------+-------+-----+-------+
|   101|          1|  John |Hyderabad|      1|2024-01-01|       1| Toyota|  Camry|30000|  30000|
|   102|          2|  Alice|  Chennai|      2|2024-01-02|       2|  Honda|  Civic|20000|  40000|
|     0|          3|Unknown|Bangalore|      0|1900-01-01|       0|Unknown|Unknown|    0|      0|
+------+-----------+-------+---------+-------+----------+--------+-------+-------+-----+-------+



In [0]:
revenue_per_customer = df.groupBy("customer_id", "name") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue")

revenue_per_customer.show()

+-----------+-------+-------------+
|customer_id|   name|total_revenue|
+-----------+-------+-------------+
|          1|  John |        30000|
|          2|  Alice|        40000|
|          3|Unknown|            0|
+-----------+-------+-------------+



2 Cars per brand


In [0]:
cars_per_brand = df.groupBy("brand") \
    .sum("quantity") \
    .withColumnRenamed("sum(quantity)", "total_cars_sold")

cars_per_brand.show()

+-------+---------------+
|  brand|total_cars_sold|
+-------+---------------+
| Toyota|              1|
|  Honda|              2|
|Unknown|              0|
+-------+---------------+



In [0]:
city_revenue = df.groupBy("city") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "city_revenue")

city_revenue.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Hyderabad|       30000|
|  Chennai|       40000|
|Bangalore|           0|
+---------+------------+



### Phase 5 – SQL Analysis


1 Top 3 per city using sql



In [0]:
# Register DataFrames as temp views
customers.createOrReplaceTempView("customers")
sales.createOrReplaceTempView("sales")
cars.createOrReplaceTempView("cars")

# Execute the SQL query
result = spark.sql("""
SELECT *
FROM (
    SELECT 
        c.city,
        c.customer_id,
        c.name,
        SUM(s.quantity * car.price) AS total_revenue,
        ROW_NUMBER() OVER (
            PARTITION BY c.city
            ORDER BY SUM(s.quantity * car.price) DESC
        ) AS rn
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    JOIN cars car ON s.car_id = car.car_id
    GROUP BY c.city, c.customer_id, c.name
) t
WHERE rn <= 3
""")

result.display()

city,customer_id,name,total_revenue,rn
Chennai,2,Alice,40000,1
Hyderabad,1,John,30000,1


In [0]:
%sql
SELECT 
    customer_id,
    COUNT(*) AS total_purchases
FROM sales
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,total_purchases


In [0]:
%sql
SELECT 
    DATE_FORMAT(sale_date, 'yyyy-MM') AS month,
    SUM(quantity * car.price) AS total_revenue
FROM sales s
JOIN cars car ON s.car_id = car.car_id
GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM')
ORDER BY month;

month,total_revenue
2024-01,85000
